In [0]:
from pyspark.sql.functions import *

In [0]:
zone_path = "/Volumes/workspace/default/nyc_data/taxi_zone_lookup.csv"

zones_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(zone_path)
)

In [0]:
display(zones_df)

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
zones_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [0]:
silver_df = spark.table(
    "workspace.nyc_taxi.silver_trips"
)

In [0]:
pickup_zone_df = (
    silver_df.join(
        zones_df,
        silver_df.PULocationID == zones_df.LocationID,
        "left"
    )
)

In [0]:
display(
    pickup_zone_df.select(
        "PULocationID",
        "Zone",
        "Borough"
    )
)

PULocationID,Zone,Borough
48,Clinton East,Manhattan
68,East Chelsea,Manhattan
141,Lenox Hill West,Manhattan
186,Penn Station/Madison Sq West,Manhattan
234,Union Sq,Manhattan
164,Midtown South,Manhattan
161,Midtown Center,Manhattan
249,West Village,Manhattan
231,TriBeCa/Civic Center,Manhattan
114,Greenwich Village South,Manhattan


In [0]:
pickup_analysis = (
    pickup_zone_df
    .groupBy(
        "Zone",
        "Borough"
    )
    .agg(
        count("*").alias("Trips"),
        sum("total_amount").alias("Revenue"),
        avg("fare_amount").alias("Average_Fare")
    )
    .orderBy(
        desc("Trips")
    )
)

In [0]:
pickup_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.nyc_taxi.gold_zone_summary"
)